<p align="center"><img src="assets/images/LOGO_Mini_Rubbish_Stelle.png" width="180"></p>

# MINI RUBBISH – Tài liệu giải thích toàn bộ project

Notebook này dành cho người **chưa biết gì** về project, và cũng chưa biết nhiều về RAG hay LLM. Đọc từ trên xuống, bạn sẽ hiểu:

1. Project giải quyết bài toán gì.
2. Các khái niệm nền: LLM, embedding, vector, chunk, RAG.
3. Mỗi thư mục và mỗi file dùng để làm gì.
4. Mỗi công nghệ được dùng ở đâu và vì sao.
5. Từng hàm trong code làm gì, nhận gì và trả về gì.
6. Cách chạy, cách đánh giá, kết quả đạt được và giới hạn.

> **Notebook này chủ yếu để đọc.** Phần lớn là chữ và đoạn code trích từ project. Chỉ mục cuối có vài ô code để thử, và các ô đó cần môi trường đã cài đặt (xem mục 16).

**Mục lục**

| # | Nội dung |
|---|---|
| 1 | Bài toán |
| 2 | Khái niệm nền cho người mới |
| 3 | Bản đồ project: thư mục và file |
| 4 | Các công nghệ được dùng |
| 5 | Luồng dữ liệu tổng thể |
| 6 | `src/config.py` – tham số |
| 7 | `src/ingest/chunking.py` – tách tài liệu |
| 8 | `src/ingest/build_index.py` – embed và lưu index |
| 9 | `src/retrieval/search.py` – tìm đoạn liên quan |
| 10 | `src/generation/answer.py` – sinh câu trả lời |
| 11 | `app/cli.py` – giao diện dòng lệnh |
| 12 | `api/main.py` và `app/web/` – API và web |
| 13 | `eval/` – đánh giá hệ thống |
| 14 | `tests/` – unit test |
| 15 | Docker và Docker Compose |
| 16 | Hướng dẫn chạy từ đầu |
| 17 | Kết quả và giới hạn |
| 18 | Bảng thuật ngữ |
| 19 | Thử nhanh (ô code) |

## 1. Bài toán

Môn **CS106 – Artificial Intelligence** có 9 bài giảng: Introduction, Uninformed Search, Informed Search, CSP & Local Search, Minimax, Expectimax, MDPs, Reinforcement Learning và Deep RL. Mỗi bài có một file tóm tắt `.txt` bằng tiếng Việt trong thư mục `OriginalDocs/`, tổng cộng khoảng 300 KB.

Sinh viên muốn hỏi nhanh, ví dụ *“A* khác Greedy Search ở điểm nào?”*, và nhận câu trả lời:
- **dựa đúng vào tài liệu môn học**, không bịa;
- **kèm nguồn**: tên file và tên mục để mở ra đối chiếu;
- **biết từ chối**: nếu tài liệu không có thông tin, hệ thống nói *“Không tìm thấy trong tài liệu”*.

**Tiêu chí xong** của project: trả lời đúng ≥ 80% bộ câu hỏi mẫu, và mọi câu trả lời đều có nguồn.

Ví dụ một lượt hỏi đáp trên CLI:
```
Câu hỏi: Thuật toán A* là gì?
A* chọn node có f(n) = g(n) + h(n) nhỏ nhất, trong đó g(n) là chi phí từ start đến n [1] ...

Nguồn:
  - 03_Informed_Search_Summary.txt > 6. A* SEARCH
```

## 2. Khái niệm nền cho người mới

### 2.1 LLM (Large Language Model)
LLM là mô hình ngôn ngữ lớn, ví dụ ChatGPT hay Gemini. Nó nhận một đoạn văn bản (gọi là **prompt**) và sinh ra văn bản tiếp theo. LLM biết rất nhiều thứ, nhưng có hai nhược điểm:
- Nó không biết nội dung riêng của môn CS106 theo đúng cách giảng viên viết.
- Nó có thể **bịa** (hallucination): trả lời trôi chảy nhưng sai.

### 2.2 RAG (Retrieval-Augmented Generation)
RAG khắc phục hai nhược điểm trên bằng hai bước:
1. **Retrieval (truy xuất):** tìm vài đoạn tài liệu liên quan nhất đến câu hỏi.
2. **Generation (sinh):** đưa các đoạn đó vào prompt và yêu cầu LLM *chỉ được* trả lời dựa trên chúng.

Có thể hình dung như một kỳ thi “mở sách”: sinh viên (LLM) được phép mở đúng vài trang tài liệu liên quan, và phải ghi rõ đã dùng trang nào.

### 2.3 Chunk
Tài liệu quá dài để đưa hết vào prompt, nên ta cắt nó thành các đoạn nhỏ gọi là **chunk**. Trong project này, mỗi chunk tương ứng với một mục của bài giảng (phần nằm giữa các dòng `=====`). Mục nào dài quá 1500 ký tự thì được cắt nhỏ thêm.

### 2.4 Embedding và vector
Máy tính không “hiểu” chữ, nên ta dùng một **embedding model** biến mỗi đoạn văn thành một dãy số, gọi là **vector**. Project dùng model `bge-m3`, cho ra vector 1024 số. Hai đoạn văn có ý nghĩa gần nhau sẽ có vector gần nhau, kể cả khi dùng từ khác nhau, ví dụ “tìm kiếm theo chiều rộng” và “BFS”.

### 2.5 Cosine similarity
Độ “gần” giữa hai vector được đo bằng **cosine**, nằm trong khoảng −1 đến 1; càng gần 1 thì càng giống nhau:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

Vector đã được **chuẩn hoá** (độ dài bằng 1) nên cosine chính là tích vô hướng $\mathbf{a}\cdot\mathbf{b}$.

### 2.6 Vector store (vector database)
Đây là nơi lưu toàn bộ vector của các chunk, kèm **metadata** (tên file, tên mục). Khi có câu hỏi, ta embed câu hỏi thành vector rồi hỏi vector store: *“cho tôi k chunk có vector gần nhất”*. Đây gọi là tìm **top-k**. Project dùng **Qdrant**.

### 2.7 Index
Index là kết quả của việc embed toàn bộ chunk và lưu vào vector store. Chỉ cần **build** một lần, xong thì truy vấn bao nhiêu lần cũng được. Index nằm ở `data/index/`.

### 2.8 LLM-as-judge
Để biết hệ thống trả lời đúng hay sai trên hàng chục câu hỏi, ta dùng một LLM khác (ChatGPT) làm “giám khảo”: so câu trả lời với đáp án mẫu. Mục 13 giải thích chi tiết.

## 3. Bản đồ project: thư mục và file

```
Mini Rubbish/
├── OriginalDocs/            # 9 file .txt bài giảng – CHỈ ĐỌC
├── data/index/              # vector index Qdrant (sinh tự động, xoá đi build lại được)
├── src/                     # lõi của hệ thống RAG
│   ├── config.py            # mọi tham số ở một chỗ
│   ├── ingest/
│   │   ├── chunking.py      # đọc file, tách chunk theo mục
│   │   └── build_index.py   # embed chunk, lưu vào Qdrant
│   ├── retrieval/
│   │   └── search.py        # tìm top-k chunk cho một câu hỏi
│   └── generation/
│       └── answer.py        # ghép prompt, gọi Gemini, trả lời kèm nguồn
├── app/
│   ├── cli.py               # hỏi đáp trên terminal
│   └── web/index.html       # giao diện chat trên trình duyệt
├── api/
│   └── main.py              # FastAPI: POST /ask, GET /toc, GET /
├── eval/
│   ├── questions/           # bộ dev: 40 câu hỏi mẫu (q01–q40.json)
│   ├── test/                # bộ test held-out: 30 câu (q101–q130.json)
│   ├── results/             # kết quả từng lần chạy đánh giá
│   ├── run_eval.py          # chạy hệ thống trên bộ câu hỏi và chấm điểm
│   └── retrieval_hits.py    # đo riêng chất lượng retrieval
├── tests/                   # unit test (pytest)
├── assets/images/, videos/  # logo, ảnh, video cho README
├── Dockerfile, docker-compose.yml, .dockerignore
├── requirements.txt, pytest.ini, .gitignore, .env
└── README.md, CLAUDE.md, TASK.md, PROGRESS.md, THEORY.md, RESULTS.md, references.bib
```

### 3.1 Thư mục

| Thư mục | Mục đích |
|---|---|
| `OriginalDocs/` | Nguồn tri thức duy nhất của hệ thống: 9 file UTF-8 tiếng Việt, chia mục bằng dòng `=====`. Không bao giờ sửa. |
| `data/index/` | Index do `build_index.py` tạo: file `storage.sqlite` chứa vector và metadata của 248 chunk. Bị `.gitignore` loại khỏi git; xoá đi thì build lại. |
| `src/` | Code lõi, chia theo ba bước của RAG: `ingest` (chuẩn bị dữ liệu), `retrieval` (tìm), `generation` (trả lời). |
| `app/` | Giao diện cho người dùng: CLI và trang web tĩnh. |
| `api/` | Web server FastAPI, nối giao diện web với `src/`. |
| `eval/` | Bộ câu hỏi mẫu, script chấm điểm và kết quả các lần chấm. |
| `tests/` | Unit test tự động, chạy bằng `pytest`. |
| `assets/` | Ảnh (logo) và video demo dùng trong README và trang web. |

### 3.2 File ở thư mục gốc

| File | Mục đích |
|---|---|
| `README.md` | Báo cáo project theo cấu trúc IMRaD: mục tiêu, phương pháp, kết quả, thảo luận, tài liệu tham khảo, hướng dẫn chạy. |
| `CLAUDE.md` | Mô tả sản phẩm, kiến trúc, lựa chọn stack và **luật làm việc** (không sửa `OriginalDocs/`, không đụng bộ test trước Task 13, đọc/ghi luôn `utf-8`...). |
| `TASK.md` | Danh sách 13 task của project, task xong được đánh `[x]`. |
| `PROGRESS.md` | Nhật ký từng phiên làm việc. Chỉ được viết thêm vào cuối, không sửa nội dung cũ. |
| `THEORY.md` | Lý thuyết và lý do đằng sau từng quyết định, trích dẫn bằng key như `[lewis2020rag]`. |
| `RESULTS.md` | Số liệu đánh giá Lần 1–5 và nhận xét. |
| `references.bib` | Danh sách tài liệu tham khảo dạng BibTeX. |
| `requirements.txt` | Các thư viện Python cần cài, kèm version cố định (pin). |
| `pytest.ini` | Cấu hình pytest: `pythonpath = .` để import được `src.*`, và `testpaths = tests`. |
| `.env` | Chứa `GOOGLE_API_KEY`. **Bí mật**: không commit, đã nằm trong `.gitignore`. |
| `.gitignore` | Những thứ git bỏ qua: `.env`, `data/index/`, `__pycache__/`, `*.pyc`. |
| `Dockerfile` | Công thức đóng gói API thành Docker image. |
| `docker-compose.yml` | Chạy cùng lúc hai container: Qdrant server và API. |
| `.dockerignore` | Những thứ không copy vào image: `.env`, `data/`, `eval/`, `tests/`, các file ghi chú... |
| `Documentation.ipynb` | Chính notebook này. |

## 4. Các công nghệ được dùng

| Công nghệ | Là gì | Dùng ở đâu trong project | Vì sao chọn |
|---|---|---|---|
| **Python 3.11** + **conda** (env `rag`) | Ngôn ngữ lập trình và công cụ quản lý môi trường | Toàn bộ code | Hệ sinh thái AI mạnh nhất; conda tách thư viện của project khỏi máy |
| **LangChain** (`langchain-core`, `langchain-huggingface`, `langchain-qdrant`, `langchain-google-genai`) | Framework ghép các thành phần LLM | `Document`, embeddings, vector store, chat model | Đổi model hoặc vector store mà ít phải sửa code; chỉ dùng phần cần thiết |
| **sentence-transformers** + **HuggingFace** | Thư viện chạy embedding model | Nạp `BAAI/bge-m3` qua `HuggingFaceEmbeddings` | Chạy local, miễn phí, không cần API |
| **BAAI/bge-m3** | Embedding model đa ngôn ngữ, vector 1024 chiều | Embed chunk và câu hỏi | Hỗ trợ tốt tiếng Việt và tiếng Anh trộn lẫn; không cần thêm prefix cho câu hỏi |
| **Qdrant** (`qdrant-client`) | Vector database | Lưu và tìm vector | Có local mode (chỉ là một thư mục, không cần server) và server mode (Docker) |
| **Gemini 3.5 Flash Lite** (Google) | LLM | Sinh câu trả lời | Có free tier; API key đặt trong `.env` |
| **python-dotenv** | Đọc file `.env` vào biến môi trường | `get_llm()` | Không hardcode API key trong code |
| **FastAPI** + **uvicorn** + **pydantic** | Web framework, web server, và thư viện kiểm tra dữ liệu | `api/main.py` | Nhanh, tự kiểm tra input, tự sinh trang tài liệu API `/docs` |
| **HTML / CSS / JavaScript** (thuần, không framework) | Giao diện web | `app/web/index.html` | Một file duy nhất, không cần build |
| **pytest** | Framework unit test | `tests/` | Chuẩn phổ biến của Python |
| **Docker** + **Docker Compose** | Đóng gói và chạy nhiều container | `Dockerfile`, `docker-compose.yml` | Chạy được trên máy khác mà không phải cài tay |
| **ChatGPT** (giao diện web) | LLM làm giám khảo | Chấm điểm trong `eval/` | Khác model với hệ thống, tránh việc model tự ưu ái câu trả lời của mình |

Version của các thư viện được pin trong `requirements.txt`:
```
langchain-google-genai==4.5.0   langchain-huggingface==1.2.3   sentence-transformers==6.3.0
langchain-qdrant==1.1.0         qdrant-client==1.20.0          python-dotenv==1.2.4
fastapi==0.165.0                uvicorn==0.54.0                pytest==9.2.0      packaging==26.3
```

## 5. Luồng dữ liệu tổng thể

Hệ thống có hai giai đoạn.

**Giai đoạn 1 – Build index (chạy một lần):**
```
OriginalDocs/*.txt
   │  chunking.py: load_chunks()  → 248 Document (nội dung + metadata {source, section})
   ▼
   │  build_index.py: get_embeddings()  → bge-m3 biến mỗi chunk thành vector 1024 số
   ▼
data/index/  (Qdrant collection "cs106")
```

**Giai đoạn 2 – Trả lời một câu hỏi (mỗi lần hỏi):**
```
Câu hỏi của người dùng
   │  search.py: search(question, k=4)  → embed câu hỏi, lấy 4 chunk gần nhất kèm điểm cosine
   ▼
   │  answer.py: điểm chunk tốt nhất < 0.5 ?  ── có ──►  "Không tìm thấy trong tài liệu"
   │                                        không
   ▼
   │  ghép prompt: SYSTEM_PROMPT + 4 chunk đánh số [1]..[4] + câu hỏi
   ▼
   │  Gemini sinh câu trả lời có ghi [n] sau mỗi ý
   ▼
   │  cited_sources(): đổi các số [n] thành "tên file > tên mục" (lấy từ metadata)
   ▼
(câu trả lời, danh sách nguồn)  →  CLI in ra  /  API trả JSON  /  web hiển thị
```

Quan hệ import giữa các module (mũi tên nghĩa là “dùng”):
```
app/cli.py ─┐
api/main.py ┼──► src/generation/answer.py ──► src/retrieval/search.py ──► src/ingest/build_index.py ──► src/ingest/chunking.py
eval/*.py ──┘                                                                                                   ▲
                                    tất cả đều đọc tham số từ src/config.py ──────────────────────────────────────┘
```

## 6. `src/config.py` – mọi tham số ở một chỗ

Luật của project: tham số hay chỉnh chỉ đặt ở **một chỗ duy nhất**, không rải khắp code. Muốn đổi `k` hay đổi model thì chỉ cần sửa file này.

```python
ROOT_DIR = Path(__file__).resolve().parent.parent   # thư mục gốc project, tính từ vị trí file
DOCS_DIR = ROOT_DIR / "OriginalDocs"

CHUNK_SIZE = 1500            # số ký tự tối đa của nội dung một chunk
EMBEDDING_MODEL = "BAAI/bge-m3"
INDEX_DIR = ROOT_DIR / "data" / "index"
COLLECTION_NAME = "cs106"
QDRANT_URL = os.getenv("QDRANT_URL")   # có thì dùng Qdrant server, không thì local mode
TOP_K = 4                    # số chunk đưa vào prompt
LLM_MODEL = "gemini-3.5-flash-lite"
GEMINI_RPM = 10              # giới hạn request/phút của free tier
MIN_SCORE = 0.5              # chunk tốt nhất dưới ngưỡng này thì từ chối luôn
NOT_FOUND_ANSWER = "Không tìm thấy trong tài liệu"
```

| Tham số | Ý nghĩa | Ghi chú |
|---|---|---|
| `ROOT_DIR` | Gốc project | Dùng `pathlib.Path` và đường dẫn tương đối, nên chạy ở máy nào cũng đúng |
| `CHUNK_SIZE` | Độ dài tối đa của chunk | 1500 ký tự, đủ cho phần lớn các mục |
| `QDRANT_URL` | Đọc từ biến môi trường | Docker Compose đặt `http://qdrant:6333`; chạy thường thì biến này rỗng |
| `TOP_K` | Số chunk lấy về | Đo thấy hit@4 bằng hit@8, nên giữ 4 cho prompt ngắn |
| `GEMINI_RPM` | Request mỗi phút | Script eval tự chờ giữa các câu để không vượt giới hạn |
| `MIN_SCORE` | Ngưỡng từ chối sớm | Chặn câu lạc đề hẳn mà không tốn một lần gọi LLM |
| `NOT_FOUND_ANSWER` | Câu từ chối chuẩn | Phải khớp từng chữ với đáp án mẫu trong `eval/` |

## 7. `src/ingest/chunking.py` – tách tài liệu thành chunk

### 7.1 Tài liệu gốc trông như thế nào
```
CS 106: ARTIFICIAL INTELLIGENCE
TÓM TẮT BÀI GIẢNG 03 — INFORMED SEARCH

Tài liệu hệ thống hóa bài giảng theo mạch: Heuristic → Greedy Search → A* Search → ...

============================================================
1. TỔNG QUAN
============================================================

Bài 02 đã giới thiệu Uninformed Search: DFS, BFS và UCS. ...

============================================================
2. ÔN LẠI SEARCH PROBLEM VÀ UCS
============================================================
...
```
Mỗi **mục** có tiêu đề kẹp giữa hai dòng `=====`. Phần trước tiêu đề đầu tiên được đặt tên mục là `"Mở đầu"`.

### 7.2 Biểu thức chính quy `SECTION_PATTERN`
```python
SECTION_PATTERN = re.compile(
    r"^={10,}[ \t]*\n(?:[ \t]*\n)?(?!=)(\S.*?)[ \t]*\n(?=={10,}[ \t]*$)",
    re.MULTILINE,
)
```
Đọc từng phần:

| Phần | Ý nghĩa |
|---|---|
| `^={10,}[ \t]*\n` | Một dòng có ít nhất 10 dấu `=` (dòng mở) |
| `(?:[ \t]*\n)?` | Có thể có một dòng trống |
| `(?!=)(\S.*?)` | Dòng tiêu đề: không bắt đầu bằng `=`; phần này được **bắt lại** làm tên mục |
| `[ \t]*\n` | Hết dòng tiêu đề |
| `(?=={10,}[ \t]*$)` | Ngay sau đó phải là dòng `=====` (dòng đóng). Đây là **lookahead**: chỉ nhìn trước, không “nuốt” dòng này |

Vì sao cần lookahead? Trong file `04-05_CSP...`, dòng `=====` đóng của mục này cũng là dòng mở của mục kế tiếp. Nếu nuốt dòng đó thì regex sẽ bỏ sót mục tiếp theo.

`SEPARATOR_LINE` khớp mọi dòng `=====` còn sót trong nội dung để xoá đi.

### 7.3 `split_sections(text) -> list[tuple[str, str]]`
- **Vào:** toàn bộ nội dung một file.
- **Ra:** danh sách cặp `(tên mục, nội dung mục)`.
- **Cách làm:**
  1. Tìm mọi tiêu đề bằng `SECTION_PATTERN.finditer`.
  2. Phần trước tiêu đề đầu tiên là mục `"Mở đầu"`.
  3. Nội dung mỗi mục kéo dài từ cuối tiêu đề này đến đầu tiêu đề kế tiếp (`zip(matches, matches[1:] + [None])` ghép mỗi tiêu đề với tiêu đề sau nó).
  4. Xoá các dòng `=====` còn sót, bỏ mục rỗng.

### 7.4 `split_long_text(text, chunk_size, separators) -> list[str]`
Cắt một mục dài thành các đoạn ≤ `chunk_size` ký tự **mà không cắt giữa câu nếu tránh được**. Thứ tự ưu tiên chỗ cắt là `SPLIT_SEPARATORS = ("\n\n", "\n", " ")`: ranh giới đoạn văn, rồi xuống dòng, rồi dấu cách.

Thuật toán **đệ quy**:
1. Nếu text đủ ngắn thì trả về nguyên.
2. Hết separator để thử (chuỗi dài liền không có dấu cách) thì cắt cứng mỗi `chunk_size` ký tự.
3. Ngược lại, tách theo separator hiện tại rồi **gom dần** các phần vào `current` chừng nào còn vừa `chunk_size`.
4. Một phần riêng lẻ mà vẫn quá dài thì gọi đệ quy với separator mịn hơn.

Ví dụ với `chunk_size = 10` và text `"aaa bbb\n\nccc ddd eee fff"`:
- Tách theo `\n\n` được `["aaa bbb", "ccc ddd eee fff"]`.
- `"aaa bbb"` (7 ký tự) vừa, giữ nguyên.
- `"ccc ddd eee fff"` (15 ký tự) quá dài, nên đệ quy với `\n` rồi với `" "`, được `["ccc ddd", "eee fff"]`.
- Kết quả: `["aaa bbb", "ccc ddd", "eee fff"]`.

Ý tưởng này giống `RecursiveCharacterTextSplitter` của LangChain, nhưng tự viết khoảng 20 dòng để không phải thêm thư viện. Hiện chưa có overlap giữa các chunk con (ghi chú `shortcut:` trong code).

### 7.5 `chunk_file(path, chunk_size) -> list[Document]`
- Đọc file với `encoding="utf-8-sig"`: giống `utf-8`, nhưng tự bỏ ký tự BOM ở đầu file nếu có.
- Với mỗi mục và mỗi đoạn con, tạo một `Document` của LangChain:
  - `page_content = "[03_Informed_Search_Summary > 6. A* SEARCH]\n<nội dung>"`: thêm tên file và tên mục ở đầu, để vector của một chunk con vẫn “biết” nó thuộc chủ đề nào (contextual header).
  - `metadata = {"source": "03_Informed_Search_Summary.txt", "section": "6. A* SEARCH"}`: dùng để trích nguồn sau này.

### 7.6 `load_chunks(docs_dir) -> list[Document]`
Đọc mọi file `*.txt` trong `OriginalDocs/` theo thứ tự tên file, rồi gộp chunk của tất cả lại. Nếu không có file nào, hàm báo `FileNotFoundError` rõ ràng thay vì âm thầm trả về danh sách rỗng. Kết quả hiện tại là **248 chunk**.

## 8. `src/ingest/build_index.py` – embed và lưu index

Chạy bằng lệnh `python -m src.ingest.build_index`. Lần đầu chạy sẽ tải model bge-m3 (khoảng 2.2 GB) về cache của HuggingFace.

### 8.1 `qdrant_location() -> dict[str, str]`
```python
return {"url": QDRANT_URL} if QDRANT_URL else {"path": str(INDEX_DIR)}
```
Hàm này quyết định kết nối Qdrant theo kiểu nào, và được **dùng chung** cho lúc build và lúc truy vấn:
- **Local mode** (`path=`): Qdrant chỉ là một thư mục trên đĩa, không cần chạy server. Tìm kiếm vét cạn, đủ nhanh với 248 chunk.
- **Server mode** (`url=`): kết nối tới Qdrant chạy trong Docker. Server dùng HNSW, thuật toán tìm gần đúng rất nhanh khi có hàng triệu vector.

### 8.2 `get_embeddings() -> HuggingFaceEmbeddings`
```python
HuggingFaceEmbeddings(model_name="BAAI/bge-m3", encode_kwargs={"normalize_embeddings": True})
```
Đây là **chỗ duy nhất** tạo embedding model. Chunk (lúc build) và câu hỏi (lúc tìm) bắt buộc phải được embed bằng *cùng một* model với *cùng một* cách chuẩn hoá; nếu không, so sánh cosine giữa chúng mất nghĩa.

### 8.3 `build_index() -> QdrantVectorStore`
1. Gọi `load_chunks()` để lấy 248 chunk.
2. Ở local mode thì tạo thư mục `data/index/` nếu chưa có.
3. `QdrantVectorStore.from_documents(...)` embed toàn bộ chunk rồi ghi vào collection `cs106`. Mặc định Qdrant dùng khoảng cách cosine.
4. `force_recreate=True` xoá collection cũ trước khi ghi, nên build lại nhiều lần cũng không bị nhân đôi chunk.

Khối `if __name__ == "__main__":` in ra số chunk đã lưu, thử tìm câu *“Thuật toán A* là gì?”*, rồi đóng client (`store.client.close()`) để tránh lỗi khi Python tắt.

## 9. `src/retrieval/search.py` – tìm đoạn liên quan

### 9.1 `get_store() -> QdrantVectorStore`
```python
@lru_cache(maxsize=1)
def get_store():
    if not QDRANT_URL and not INDEX_DIR.exists():
        raise FileNotFoundError("Chưa có index ... Chạy: python -m src.ingest.build_index")
    client = QdrantClient(**qdrant_location())
    if not client.collection_exists(COLLECTION_NAME):
        client.close(); raise FileNotFoundError(...)
    return QdrantVectorStore(client=client, collection_name=COLLECTION_NAME, embedding=get_embeddings())
```
- **`@lru_cache(maxsize=1)`**: lần gọi đầu tạo store, các lần sau trả lại đúng object đó. Có hai lý do:
  1. Qdrant local mode **khoá** thư mục index, nên mở client thứ hai trong cùng tiến trình sẽ báo lỗi.
  2. Nạp bge-m3 mất vài giây và vài GB RAM; mỗi câu hỏi nạp lại một lần thì quá chậm.
- **Kiểm tra lỗi trước:** báo rõ nếu chưa build index, kèm lệnh cần chạy. Thư mục được kiểm tra *trước* khi mở client, vì local mode sẽ tự tạo một thư mục rỗng nếu chưa có.

### 9.2 `search(question, k=TOP_K) -> list[tuple[Document, float]]`
```python
return get_store().similarity_search_with_score(question, k=k)
```
Embed câu hỏi rồi trả về `k` cặp `(chunk, điểm cosine)`, điểm cao nhất đứng trước. Ví dụ:
```
[(Document(page_content="[03_Informed_Search_Summary > 6. A* SEARCH]\n...", metadata={...}), 0.776),
 (Document(...), 0.701), ...]
```
Điểm số được giữ lại để bước generation so với `MIN_SCORE`.

## 10. `src/generation/answer.py` – sinh câu trả lời kèm nguồn

### 10.1 `SYSTEM_PROMPT`
Đây là lời dặn cho LLM, được gửi kèm mọi câu hỏi. Các ý chính:
- Chỉ trả lời dựa trên các đoạn tài liệu được cung cấp, không dùng kiến thức bên ngoài.
- Trả lời bằng tiếng Việt, **đầy đủ**: nêu mọi định nghĩa, công thức, điều kiện, ví dụ số; giữ nguyên ký hiệu.
- Câu hỏi so sánh thì trình bày đủ từng vế.
- Tài liệu chỉ có một phần thông tin thì vẫn trả lời phần đó.
- Sau mỗi ý, ghi số đoạn đã dùng, ví dụ `[1]` hoặc `[2][3]`.
- Không có thông tin thì chỉ viết đúng câu `NOT_FOUND_ANSWER`.

Bản đầu của prompt yêu cầu “ngắn gọn”, khiến model bỏ mất công thức và điều kiện. Đổi sang “đầy đủ” nâng độ chính xác trên bộ dev từ 35% lên 58% (xem `RESULTS.md` Lần 3–4).

### 10.2 `get_llm() -> BaseChatModel`
- Gọi `load_dotenv(ROOT_DIR / ".env")` để đọc `GOOGLE_API_KEY` vào biến môi trường.
- Thiếu key thì báo `RuntimeError` kèm hướng dẫn sửa.
- Tạo `ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0)`. Model Flash Lite thực tế bỏ qua `temperature`, nên câu trả lời có thể khác nhau đôi chút giữa các lần chạy.
- Có `@lru_cache`, nên chỉ tạo client một lần.

### 10.3 `source_label(doc)` và `format_context(docs)`
- `source_label` trả về `"tên file > tên mục"`.
- `format_context` đánh số các chunk để LLM trích lại bằng số:
```
[1] (03_Informed_Search_Summary.txt > 6. A* SEARCH)
[03_Informed_Search_Summary > 6. A* SEARCH]
A* kết hợp UCS và Greedy ...

[2] (...)
```

### 10.4 `cited_sources(answer_text, docs) -> list[str]`
Hàm này biến các số `[n]` trong câu trả lời thành danh sách nguồn:
1. Câu trả lời là câu từ chối thì không có nguồn.
2. Dùng regex `\[(\d+)\]` tìm mọi số được trích, giữ lại các chunk tương ứng.
3. LLM quên ghi số nào thì dùng tất cả chunk đã đưa vào, để câu trả lời nào cũng có nguồn.
4. `dict.fromkeys(...)` bỏ nguồn trùng nhưng giữ thứ tự.

**Điểm quan trọng:** tên nguồn lấy từ **metadata** do code tạo, không lấy chữ do LLM tự viết. Vì vậy LLM không thể bịa ra tên file hay tên mục không tồn tại.

### 10.5 `answer(question, k) -> tuple[str, list[str]]`
Đây là hàm chính mà CLI, API và eval đều gọi:
```python
results = search(question, k=k)
if not results or results[0][1] < MIN_SCORE:      # lớp chặn 1: retrieval
    return NOT_FOUND_ANSWER, []
docs = [doc for doc, _ in results]
messages = [("system", SYSTEM_PROMPT),
            ("human", f"Tài liệu:\n{format_context(docs)}\n\nCâu hỏi: {question}")]
text = get_llm().invoke(messages).text.strip()   # lớp chặn 2: prompt bắt LLM từ chối nếu không có
return text, cited_sources(text, docs)
```

Câu hỏi ngoài phạm vi bị chặn ở **hai lớp**:
1. Chunk tốt nhất có điểm < 0.5: trả lời “không tìm thấy” ngay, không tốn lần gọi Gemini.
2. Điểm đủ cao nhưng nội dung không trả lời được câu hỏi: prompt yêu cầu LLM từ chối.

Đo thực tế thấy điểm của câu ngoài phạm vi (0.40–0.53) chồng lên điểm câu trong phạm vi (thấp nhất 0.52). Vì vậy lớp 2 mới là lớp chặn chính.

## 11. `app/cli.py` – hỏi đáp trên terminal

Chạy bằng `python -m app.cli`. Thêm `--debug` để in điểm cosine của các chunk tìm được.

| Hàm | Việc làm |
|---|---|
| `format_reply(text, sources)` | Ghép câu trả lời với dòng `Nguồn:` và danh sách nguồn (nếu có) |
| `format_debug(question)` | Gọi `search()` rồi in điểm của từng chunk, so với `MIN_SCORE` |
| `main()` | Vòng lặp: đọc câu hỏi, gọi `answer()`, in kết quả |

Chi tiết trong `main()`:
- `argparse` đọc tham số dòng lệnh `--debug`.
- Tắt thanh tiến trình của HuggingFace và cảnh báo “sampling parameter” của Gemini cho gọn màn hình.
- Gõ `exit`, `quit`, `thoát`, Ctrl+C hoặc Ctrl+D để thoát.
- Xử lý lỗi rõ ràng:
  - Gemini quá tải hoặc hết quota: in thông báo và cho hỏi tiếp.
  - Thiếu index hoặc thiếu API key: in lỗi rồi thoát.
- `finally:` đóng Qdrant client nếu đã mở, tránh lỗi khi Python tắt.

## 12. `api/main.py` và `app/web/` – API và giao diện web

Chạy bằng `uvicorn api.main:app`, rồi mở `http://127.0.0.1:8000`.

### 12.1 Khái niệm
- **API**: các “cửa” mà chương trình khác (ở đây là trang web) gọi qua HTTP.
- **FastAPI**: thư viện định nghĩa các cửa đó bằng hàm Python có decorator như `@app.post("/ask")`.
- **uvicorn**: web server thực sự chạy ứng dụng FastAPI.
- **pydantic**: khai báo kiểu dữ liệu vào/ra; FastAPI dựa vào đó để tự kiểm tra input và tự trả lỗi 422 khi input sai.

### 12.2 Các model dữ liệu
```python
class AskRequest(BaseModel):     # body của POST /ask
    question: Annotated[str, StringConstraints(strip_whitespace=True, min_length=1, max_length=1000)]
class AskResponse(BaseModel):    # kết quả
    answer: str
    sources: list[str]
class Lecture(BaseModel):        # một phần tử của mục lục
    source: str
    sections: list[str]
```

### 12.3 `lifespan`
Hàm này chạy **lúc server khởi động**: gọi `get_store()` và `get_llm()` trước. Nhờ vậy:
- thiếu index hoặc thiếu key thì server báo lỗi ngay khi bật;
- người hỏi đầu tiên không phải chờ nạp model.

Lúc server tắt, hàm đóng Qdrant client.

### 12.4 Các endpoint

| Endpoint | Hàm | Trả về |
|---|---|---|
| `GET /` | `home()` | File `app/web/index.html` (trang chat) |
| `GET /toc` | `table_of_contents()` | Mục lục: mỗi bài và danh sách mục theo thứ tự trong tài liệu. Có `@lru_cache` vì tài liệu không đổi khi server đang chạy |
| `POST /ask` | `ask(request)` | `{"answer": "...", "sources": ["file > mục", ...]}` |
| `GET /files/<tên file>` | `StaticFiles` | Tải file `.txt` gốc. `StaticFiles` chặn sẵn đường dẫn như `../.env`, nên không lộ được file ngoài thư mục |
| `GET /assets/...` | `StaticFiles` | Logo |
| `GET /docs` | (FastAPI tự sinh) | Trang Swagger để thử API |

`ask()` đổi lỗi Python thành mã HTTP:

| Lỗi | Mã HTTP | Ý nghĩa |
|---|---|---|
| Input sai (rỗng, dài quá 1000 ký tự) | 422 | pydantic tự trả |
| `GoogleRateLimitError` | 429 | Gemini hết quota |
| `GoogleAPIError` | 502 | Gemini lỗi |
| `FileNotFoundError`, `RuntimeError` | 503 | Thiếu index hoặc thiếu API key |

`ask` được khai báo bằng `def` thường, không phải `async def`, vì `answer()` chạy blocking (đợi Gemini). Với hàm thường, FastAPI tự đẩy sang threadpool nên server không bị treo.

Chỉ chạy **một worker**, vì Qdrant local mode khoá thư mục index.

### 12.5 `app/web/index.html`
Một file HTML duy nhất gồm CSS và JavaScript thuần (khoảng 300 dòng):
- **Bố cục:** header có logo, khung chat bên trái, mục lục bài giảng bên phải. Màu sắc có chế độ sáng/tối theo hệ điều hành (`prefers-color-scheme`).
- **`ask(question)`:** gửi `fetch("/ask", {method: "POST", ...})`, hiện câu trả lời và danh sách nguồn (`addMessage`, `showSources`), báo lỗi nếu API trả mã lỗi.
- **`loadToc()` / `renderLecture()`:** gọi `GET /toc` để vẽ mục lục. Bấm một mục sẽ điền sẵn câu hỏi về mục đó; mỗi bài có link tải file gốc qua `/files/...`.
- **`parseLecture` / `lectureLabel`:** đổi tên file như `03_Informed_Search_Summary.txt` thành nhãn dễ đọc.

## 13. `eval/` – đánh giá hệ thống

### 13.1 Định dạng một câu hỏi mẫu
Mỗi câu hỏi là một file JSON, ví dụ `eval/questions/q31.json`:
```json
{
  "id": 31,
  "type": "không dấu",
  "question": "su khac nhau giua bfs va dfs la gi",
  "expected_answer": "DFS chọn deepest node, fringe giống stack, ... BFS chọn shallowest node, ...",
  "source_file": "02_Uninformed_Search_Summary.txt",
  "section": "15. DFS VS BFS"
}
```
- `source_file` có ba dạng: `null` (câu ngoài phạm vi), một tên file, hoặc một list (câu cần nhiều bài).
- `type` có các giá trị: thường, không dấu, tiếng Anh, trộn Anh-Việt, sai chính tả, ngắn/mơ hồ, nhiều bài.

### 13.2 Hai bộ câu hỏi
- **Dev** (`eval/questions/`, 40 câu): dùng thoải mái để tinh chỉnh hệ thống.
- **Test** (`eval/test/`, 30 câu): **held-out**, chỉ chạy **một lần** ở cuối project, không tinh chỉnh sau khi xem kết quả. Lý do: nếu vừa nhìn điểm vừa sửa hệ thống trên cùng một bộ, điểm sẽ lạc quan dần (overfit) và không còn phản ánh câu hỏi mới.

### 13.3 `run_eval.py` – quy trình chấm
```
1. python -m eval.run_eval [--split test]
      → hệ thống trả lời từng câu → ghi eval/results/<run>.jsonl
      → xuất <run>.judge_prompt.txt và <run>.judge_input.json
2. Người dùng dán prompt + upload file lên ChatGPT → tải về verdicts.json
3. python -m eval.run_eval --resume <run>.jsonl --verdicts <run>.verdicts.json
      → gắn kết quả chấm, in bảng tổng kết, ghi <run>.summary.txt
```

Các hàm chính:

| Hàm | Việc làm |
|---|---|
| `Verdict` (pydantic) | Kết quả chấm một câu: `missing_points`, `wrong_points`, `faithful`, `reason`. Thuộc tính `correct` chỉ đúng khi **cả hai danh sách đều rỗng**; code quyết định đúng/sai, không để judge tự chọn |
| `wait_for_rate_limit()` | Chờ giữa các lần gọi Gemini để không vượt `GEMINI_RPM` |
| `load_questions(split)` | Đọc mọi `q*.json` của bộ dev hoặc test |
| `expected_files(q)` | Chuẩn hoá `source_file` thành list (`null` thành `[]`) |
| `is_refusal(text)` | Câu trả lời có phải câu từ chối không |
| `score(...)` | Chấm một câu: đúng/sai, nguồn đúng (đủ mọi file cần có), retrieval hit |
| `summarize(records)` | Tính các tỉ lệ và độ chính xác theo loại câu |
| `judge(...)` | Chấm tự động bằng Gemini (`--judge gemini`), chỉ để chạy nhanh vì bị thiên lệch |
| `evaluate(q, ...)` | Chạy hệ thống trên một câu, lưu kèm context để judge đánh giá độ trung thực |
| `export_for_manual_judge(...)` | Xuất file cho ChatGPT, chỉ gồm câu cần chấm (trong phạm vi và không bị từ chối) |
| `apply_verdicts(...)` | Gắn verdicts vào kết quả; thiếu id nào thì báo lỗi, không đoán |
| `read_records(path)` | Đọc file `.jsonl` (mỗi dòng một JSON), dùng cho `--resume` |
| `main()` | Đọc tham số `--split`, `--judge`, `--resume`, `--verdicts`, chạy quy trình. Gặp lỗi Gemini giữa chừng thì in lệnh chạy tiếp |

Các chỉ số được in ra:

| Chỉ số | Cách tính |
|---|---|
| Độ chính xác | Số câu `correct` trên tổng số câu. Câu ngoài phạm vi đúng khi bị từ chối |
| Nguồn đúng | Câu trong phạm vi có trích đủ các file cần có |
| Câu trả lời có nguồn | Trong các câu không bị từ chối, bao nhiêu câu có ít nhất một nguồn |
| Từ chối đúng / oan | Câu ngoài phạm vi bị từ chối / câu trong phạm vi bị từ chối nhầm |
| Trung thực | Judge xác nhận mọi ý trong câu trả lời đều có trong context |
| Retrieval hit@k | Mọi file cần có đều nằm trong top-k |

### 13.4 `retrieval_hits.py` – đo riêng retrieval (không gọi Gemini)
- **File hit@k:** mọi file cần có đều nằm trong top-k.
- **Section hit@k:** mọi cặp (file, mục) cần có đều nằm trong top-k. Chặt hơn file hit, vì mỗi file có hơn 20 mục.
- **MRR@k** (Mean Reciprocal Rank): trung bình của $1/\text{hạng}$ của chunk đúng đầu tiên, bằng 0 nếu không có trong top-k. Chunk đúng ở hạng 1 được 1 điểm, hạng 2 được 0.5, và cứ thế giảm.

$$\text{MRR@}k = \frac{1}{|Q|}\sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

Các hàm phụ:
- `expected_pairs`: cặp (file, mục) cần có.
- `rank_of`: hạng của một phần tử trong danh sách.
- `is_hit`: mọi mục tiêu đều trong top-k.
- `reciprocal_rank`: $1/$ hạng tốt nhất.

Script chạy với k = 4, 6, 8, 10 và in thêm điểm top-1 thấp nhất, để xem `MIN_SCORE` có tách được câu ngoài phạm vi không.

### 13.5 `eval/results/`
Mỗi lần chạy có một tiền tố thời gian, ví dụ `20261119-024609`; bộ test có thêm tiền tố `test-`. Mỗi lần chạy gồm các file:

| Đuôi | Nội dung |
|---|---|
| `.jsonl` | Từng câu: câu hỏi, câu trả lời, nguồn, context, kết quả chấm |
| `.judge_prompt.txt` | Prompt dán vào ChatGPT |
| `.judge_input.json` | Dữ liệu upload lên ChatGPT |
| `.verdicts.json` | Kết quả ChatGPT trả về |
| `.summary.txt` | Bảng tổng kết |

## 14. `tests/` – unit test

Chạy bằng `pytest tests/`; hiện có 49 test, tất cả đều pass. Unit test là các hàm nhỏ tự kiểm tra code có chạy đúng không, để mỗi lần sửa code ta biết ngay mình có làm hỏng gì.

Nhiều test dùng `monkeypatch` để **thay thế** hàm thật bằng hàm giả, ví dụ giả `search()` hay giả Gemini. Nhờ vậy test chạy nhanh, không tốn quota và không cần mạng.

| File | Kiểm tra gì |
|---|---|
| `test_chunking.py` (7) | Tách đúng tiêu đề và nội dung mục; chunk không quá `CHUNK_SIZE` và không cắt giữa từ; cắt cứng chuỗi dài liền; chunk của tài liệu thật có đủ metadata; báo lỗi khi thư mục rỗng; xử lý BOM; **nhãn `section` trong eval khớp với mục thật của tài liệu** |
| `test_retrieval.py` (7) | `search` trả đúng k chunk, sắp xếp theo điểm; tìm được đúng bài; báo lỗi rõ khi thiếu index hoặc server chưa có collection; chọn url/path đúng; index khớp với tài liệu hiện tại; chất lượng retrieval không bị giảm |
| `test_generation.py` (6) | Đánh số context; chỉ giữ nguồn được trích và bỏ trùng; không có số `[n]` thì dùng mọi nguồn; câu từ chối không có nguồn; dưới `MIN_SCORE` thì **không gọi LLM**; trả đúng text và nguồn |
| `test_api.py` (9) | `/ask` trả answer + sources; câu ngoài phạm vi không có nguồn; input sai trả 422; lỗi Gemini và lỗi thiếu index đổi đúng mã 429/502/503; `/toc` đủ 9 bài; `/` trả trang chat; tải được file gốc; **không thoát ra ngoài thư mục được** (`../.env`); logo hiển thị |
| `test_cli.py` (3) | Định dạng câu trả lời có và không có nguồn; chế độ debug in điểm |
| `test_eval.py` (12) | Logic chấm điểm: verdict sai khi thiếu hoặc sai ý; gắn verdicts và báo lỗi khi thiếu id; câu nhiều bài cần đủ file; từ chối oan bị tính sai; file câu hỏi đủ trường; **bộ test tách biệt với bộ dev** |
| `test_retrieval_hits.py` (4) | Các hàm tính hạng, hit@k và MRR |

Một số test (retrieval, index) cần index thật đã được build và model bge-m3 đã tải về.

## 15. Docker và Docker Compose

**Docker** đóng gói ứng dụng cùng mọi thư viện vào một **image**; chạy image đó ta được một **container**, giống một máy ảo nhẹ. **Docker Compose** chạy nhiều container cùng lúc từ một file cấu hình.

### 15.1 `Dockerfile` (image của API)
```dockerfile
FROM python:3.11-slim                       # nền: Python 3.11 bản gọn
WORKDIR /app
RUN pip install torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu   # torch bản CPU, nhẹ hơn bản CUDA vài GB
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY src/ src/                              # tương tự cho api/, app/, OriginalDocs/, assets/
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
```
(Đây là bản rút gọn; file thật có thêm cờ `--no-cache-dir` và `EXPOSE 8000`.)

### 15.2 `docker-compose.yml`

| Service | Vai trò | Chi tiết |
|---|---|---|
| `qdrant` | Qdrant server | Image `qdrant/qdrant:v1.20.2` (khớp `qdrant-client` 1.20), cổng 6333, dashboard ở `http://localhost:6333/dashboard`, dữ liệu lưu trong volume `qdrant_data` |
| `api` | FastAPI | Build từ `Dockerfile`, cổng 8000, đọc key từ `.env` (không copy key vào image), đặt `QDRANT_URL=http://qdrant:6333`, volume `hf_cache` giữ bge-m3 để không phải tải lại, `restart: on-failure:5` phòng khi API bật trước Qdrant |

Vì Compose đặt biến `QDRANT_URL`, code tự chuyển sang server mode (`qdrant_location()`). Chạy trực tiếp trên máy thì không có biến này, nên vẫn dùng local `data/index`.

**Trạng thái hiện tại:** container Qdrant đã chạy được. Bước build index trong Docker phải dừng giữa chừng vì máy gần hết RAM, nên API chưa được chạy end-to-end trong Docker.

## 16. Hướng dẫn chạy từ đầu

Các lệnh dưới đây chạy trong terminal, ở thư mục gốc project.

```bash
# 1. Tạo môi trường và cài thư viện (một lần)
conda create -n rag python=3.11
conda activate rag
pip install -r requirements.txt

# 2. Tạo file .env ở thư mục gốc, chứa một dòng:
#    GOOGLE_API_KEY=<key lấy ở Google AI Studio>

# 3. Build index (một lần; lần đầu tải bge-m3 ~2.2 GB)
python -m src.ingest.build_index

# 4a. Hỏi đáp trên terminal
python -m app.cli            # thêm --debug để xem điểm cosine

# 4b. Hoặc chạy web/API rồi mở http://127.0.0.1:8000
uvicorn api.main:app

# 5. Chạy unit test
pytest tests/

# 6. Đánh giá trên bộ dev
python -m eval.run_eval
python -m eval.retrieval_hits
```

**Lưu ý:**
- Không chạy CLI và API cùng lúc, vì Qdrant local mode chỉ cho một tiến trình mở `data/index`.
- Free tier của Gemini giới hạn khoảng 10 request/phút. Gặp lỗi 429 thì chờ một lúc rồi thử lại.
- Muốn đổi tham số (`TOP_K`, `CHUNK_SIZE`, model) thì sửa `src/config.py`. Đổi `CHUNK_SIZE` hoặc model embedding thì phải build lại index.

## 17. Kết quả và giới hạn

| Lần | Thay đổi | Bộ | Độ chính xác | Có nguồn |
|---|---|---|---|---|
| 1 | Baseline, judge Gemini dễ dãi | dev | 98% | 100% |
| 2 | Judge Gemini chặt (đếm từng ý) | dev | 58% | 100% |
| 3 | Judge ChatGPT chặt | dev | 35% | 100% |
| 4 | Prompt “đầy đủ” thay vì “ngắn gọn” | dev | 58% | 100% |
| **5** | **Không đổi, held-out** | **test** | **60%** | **100%** |

- Retrieval rất tốt: section hit@4 = 92%, MRR = 0.907 trên dev; hit@4 = 27/27 trên test.
- **Tiêu chí có nguồn: đạt (100%). Tiêu chí ≥ 80% đúng: chưa đạt (60%).**
- Nút thắt là **model sinh câu trả lời** (Gemini Flash Lite). Câu trả lời thường thiếu ý so với đáp án mẫu, dù ý bị thiếu có sẵn trong context; hiếm khi trả lời sai.
- Cùng một bộ câu trả lời có thể được chấm 98% hay 35% tuỳ judge và tiêu chí. Vì vậy cần judge khác model và tiêu chí chặt.
- Giới hạn khác:
  - Mỗi loại câu khó chỉ có 1–3 câu, nên tỉ lệ theo loại dao động mạnh.
  - Đáp án mẫu do người làm project viết.
  - Docker mới chạy được phần Qdrant.

Hướng cải thiện (phải đánh giá lại trên dev, vì bộ test đã được dùng):
1. Dùng model sinh câu trả lời mạnh hơn.
2. Cho prompt liệt kê đủ ý từ context trước rồi mới viết câu trả lời.
3. Tách câu hỏi nhiều bài thành các câu con trước khi truy xuất.

Chi tiết: `RESULTS.md`, `README.md` (mục 5–6) và `THEORY.md`.

## 18. Bảng thuật ngữ

| Thuật ngữ | Giải thích ngắn |
|---|---|
| RAG | Truy xuất tài liệu liên quan rồi cho LLM trả lời dựa trên đó |
| LLM | Mô hình ngôn ngữ lớn sinh văn bản (Gemini, ChatGPT) |
| Prompt / system prompt | Văn bản gửi cho LLM / phần lời dặn cố định đi kèm mọi câu hỏi |
| Chunk | Một đoạn nhỏ của tài liệu, là đơn vị được embed và truy xuất |
| Embedding / vector | Dãy số biểu diễn ý nghĩa của một đoạn văn |
| Cosine similarity | Độ giống nhau của hai vector, từ −1 đến 1 |
| Vector store / Qdrant | Cơ sở dữ liệu lưu và tìm vector gần nhất |
| Index / collection | Toàn bộ vector đã lưu / tên “bảng” trong Qdrant (`cs106`) |
| Top-k | k kết quả có điểm cao nhất |
| Metadata | Thông tin kèm chunk: `source` (file), `section` (mục) |
| Hallucination | LLM bịa thông tin không có trong tài liệu |
| Faithful (trung thực) | Mọi ý trong câu trả lời đều có trong context |
| HNSW | Thuật toán tìm vector gần đúng rất nhanh, dùng ở Qdrant server |
| Local mode / server mode | Qdrant chạy trong tiến trình Python (thư mục) / chạy thành server riêng |
| LLM-as-judge | Dùng LLM để chấm câu trả lời của hệ thống |
| Self-preference bias | Judge ưu ái câu trả lời do chính model của nó sinh ra |
| Dev / held-out test | Bộ dùng để tinh chỉnh / bộ giữ riêng, chỉ chạy một lần cuối cùng |
| hit@k, MRR | Đáp án có trong top-k không / chunk đúng đứng ở hạng cao hay thấp |
| API / endpoint | Giao diện để chương trình khác gọi / một “cửa” cụ thể như `POST /ask` |
| `lru_cache` | Decorator Python ghi nhớ kết quả hàm, dùng để chỉ tạo model/client một lần |
| `monkeypatch` | Công cụ pytest thay hàm thật bằng hàm giả khi test |
| `.env` | File chứa biến bí mật như API key, không commit lên git |
| Docker image / container | Gói ứng dụng / bản đang chạy của gói đó |

## 19. Thử nhanh (ô code)

Các ô dưới đây dùng lại đúng các hàm của project. Điều kiện để chạy:
- Kernel là conda env `rag`.
- Notebook mở từ thư mục gốc project.
- Đã build index (ô 2 trở đi).
- Có `GOOGLE_API_KEY` (ô 3).

Đang chạy CLI hoặc API thì **tắt trước**, vì Qdrant local mode chỉ cho một tiến trình mở index.

**Ô 1 – xem chunking** (không cần index, không gọi API):

In [ ]:
from src.ingest.chunking import load_chunks

chunks = load_chunks()
print(f"Tổng số chunk: {len(chunks)}")
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

**Ô 2 – thử retrieval** (cần index; nạp bge-m3 mất vài giây):

In [ ]:
from src.retrieval.search import search

for doc, score in search("Thuật toán A* là gì?", k=4):
    print(f"{score:.3f}  {doc.metadata['source']} > {doc.metadata['section']}")

**Ô 3 – hỏi đáp đầy đủ** (gọi Gemini, tốn một request của free tier):

In [ ]:
from src.generation.answer import answer

text, sources = answer("Sự khác nhau giữa SARSA và Q-Learning là gì?")
print(text)
print("\nNguồn:", *sources, sep="\n  - ")

**Ô 4 – đóng index khi xong**, để CLI hoặc API mở lại được:

In [ ]:
from src.retrieval.search import get_store

if get_store.cache_info().currsize:
    get_store().client.close()